<h1>Chapter 8 - Semantic Search and Retrieval-Augmented Generation</h1>
<i>Exploring a vital part of LLMs, search.</i>

<a href="https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961"><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="https://www.oreilly.com/library/view/hands-on-large-language/9781098150952/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/HandsOnLLM/Hands-On-Large-Language-Models"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter08/Chapter%208%20-%20Semantic%20Search.ipynb)

---

This notebook is for Chapter 8 of the [Hands-On Large Language Models](https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961) book by [Jay Alammar](https://www.linkedin.com/in/jalammar) and [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/).

---

<a href="https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961">
<img src="https://raw.githubusercontent.com/HandsOnLLM/Hands-On-Large-Language-Models/main/images/book_cover.png" width="350"/></a>


### [OPTIONAL] - Installing Packages on <img src="https://colab.google/static/images/icons/colab.png" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** the following codeblock to install the dependencies for this chapter:

---

💡 **NOTE**: We will want to use a GPU to run the examples in this notebook. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**.

---


In [1]:
# %%capture
# !pip install langchain==0.2.5 faiss-cpu==1.8.0 cohere==5.5.8 langchain-community==0.2.5 rank_bm25==0.2.2 sentence-transformers==3.0.1
# !pip install llama-cpp-python==0.2.78  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

## IMPORTANT: Make sure to restart the session after installing the packages above.

# Dense Retrieval Example


## 1. Getting the text archive and chunking it


In [2]:
import os
import cohere
from dotenv import load_dotenv

# Carga variables de entorno from .env
load_dotenv()

# Retrieve Cohere API key
api_key = os.getenv("COHERE_API_KEY")
# Create and retrieve a Cohere API key from os.cohere.ai
co = cohere.Client(api_key)

In [3]:
text = """
Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan.
It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine.
Set in a dystopian future where humanity is struggling to survive, the film follows a group of astronauts who travel through a wormhole near Saturn in search of a new home for mankind.

Brothers Christopher and Jonathan Nolan wrote the screenplay, which had its origins in a script Jonathan developed in 2007.
Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar.
Cinematographer Hoyte van Hoytema shot it on 35 mm movie film in the Panavision anamorphic format and IMAX 70 mm.
Principal photography began in late 2013 and took place in Alberta, Iceland, and Los Angeles.
Interstellar uses extensive practical and miniature effects and the company Double Negative created additional digital effects.

Interstellar premiered on October 26, 2014, in Los Angeles.
In the United States, it was first released on film stock, expanding to venues using digital projectors.
The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014.
It received acclaim for its performances, direction, screenplay, musical score, visual effects, ambition, themes, and emotional weight.
It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics. Since its premiere, Interstellar gained a cult following,[5] and now is regarded by many sci-fi experts as one of the best science-fiction films of all time.
Interstellar was nominated for five awards at the 87th Academy Awards, winning Best Visual Effects, and received numerous other accolades"""

#text -> es simplemente un str largo con información sobre Interstellar.

# Split into a list of sentences
texts = text.split('.')
# divide el string cada vez que encuentra un punto y lo mete en una lista

# Clean up to remove empty spaces and new lines
texts = [t.strip(' \n') for t in texts]
# elimina espacios y saltos de página
# strip() -> elimina caracteres de los extremes del string

## 2. Embedding the Text Chunks


In [4]:
import numpy as np

# Get the embeddings -> usamos el modelo de Cohere
response = co.embed(
  texts=texts,
  model="embed-english-v3.0",
  input_type="search_document",
).embeddings

embeds = np.array(response)
# convierte los embeddings que devuelve Cohere en un array de NumPy.

print(embeds.shape)

# La ventaja de convertirlo a NumPy es que ahora puedes hacer fácilmente operaciones matemáticas con esos vectores, 
# que es justo lo que necesitaremos para buscar similitudes.

(15, 1024)


Tenemos 15 vectores. Cada uno con un tamaño de 1024

Solo hemos preparado la "base de datos vectorial" sobre la que harás la búsqueda. La siguiente pieza será convertir también la query en un embedding y preguntarle a FAISS cuáles de estos vectores están más cerca

## 3. Building The Search Index


In [5]:
import faiss
# FAISS -> librería para búsqueda eficiente de vectores similares 

dim = embeds.shape[1]
# obtenemos la segunda dimensión de la shape de nuestros embeddings -> la dimensión -> 1024
# FAISS necesita saber que va a trabajar con vectores de 1024 dimensiones.

index = faiss.IndexFlatL2(dim)
# Creamos un índice que buscará los vectores utilizando distancia euclídea al cuadrado (L2).

index.add(np.float32(embeds))
# añade nuestros embeddings al índice
# np.float32(embeds) -> los convierte a números de 32 bits, que es el formato que espera FAISS

## 4. Search the index


In [6]:
import pandas as pd

def search(query, number_of_results=3): # recibe una query y devuelve 3 resultados cercanos

  # 1. Get the query's embedding
  query_embed = co.embed(texts=[query],
                model="embed-english-v3.0",
                input_type="search_query",).embeddings[0]
    #[0] -> porque pasamos una lista de texto a Cohere, aunque esa lista tenga sólo un elemento

  # 2. Retrieve the nearest neighbors
  distances , similar_item_ids = index.search(np.float32([query_embed]), number_of_results)
    # Ésta es la línea en la que FAISS realiza realmente la búsqueda
    # np.float32([query_embed]) -> query_embed -> vector de (1024,) pero FAISS necesita matriz -> [query_embed] ->(1,2024) -> 1 query
    # index.search(..., number_of_results) -> Busca en index los number_of_results vectores más próximos a esta query.
    # estructura -> D, I = index.search(...)
    # Devuelve 2 matrices:
        # distances -> tiene las distancias de la query a los distintos textos ordenado de más a menos cercano
        # similar_item_ids -> ID de los docuembhtos cercanos. Y los IDs permiten volver al texto original: text[1], text[4]...

  # 3. Format the results
  texts_np = np.array(texts) # Convert texts list to numpy for easier indexing
  # Pasamos las distintas frases a array de numpy -> Porque NumPy permite hacer algo muy cómodo: seleccionar varios elementos pasando 
  # una lista/array de índices.

    
  results = pd.DataFrame(data={'id': similar_item_ids[0],
                              'texts': texts_np[similar_item_ids[0]],
                              'distance': distances[0]})
  # Crea un DF
  # Empareja ambas cosas: los textos y similar_items_ids[0] que sirven de índices para text_np

  # 4. Print and return the results
  print(f"Query:'{query}'\nNearest neighbors:")
  # Imprime la Query + Nearest neighbors: y después return -> Dataframe textos - Distancias
  return results

In [7]:
# escribimos una query y aplicamos la función search()
query = "how precise was the science"
results = search(query)
results

Query:'how precise was the science'
Nearest neighbors:


,id,texts,distance
0,12,It has also received praise from many astronom...,1.200760
1,4,Caltech theoretical physicist and 2017 Nobel l...,1.370212
2,7,Interstellar uses extensive practical and mini...,1.580641


In [8]:
# ahora vamos a construir una búsqueda más tradicional usando BM25
# BM25 no trabaja con embeddings sino con palabras/términos y sus frecuencias
# Por eso necesitamos convertir previamente el docuemnto en tokens

from rank_bm25 import BM25Okapi
from sklearn.feature_extraction import _stop_words
import string

def bm25_tokenizer(text): # función para preparar/tokenizar el texto antes de pasárselo a BM25
    tokenized_doc = [] # lista donde irán los tokens válidos
    for token in text.lower().split(): # pasa texto a minúscualas y separa por espacios
        token = token.strip(string.punctuation) # elimina signos de puntuación de los extremos de cada token

        if len(token) > 0 and token not in _stop_words.ENGLISH_STOP_WORDS:
            tokenized_doc.append(token)
            # Evita añadir tokens que hayan quedado vacíos.
            # Elimina las stop words, palabras muy frecuentes que normalmente aportan poca información para una búsqueda
    return tokenized_doc

In [9]:
from tqdm import tqdm

tokenized_corpus = [] # creas una lista vacía donde guardarás todos los documentos tokenizados
for passage in tqdm(texts): # recorres cada frase de texts
    tokenized_corpus.append(bm25_tokenizer(passage)) # aplica la función b,25_tokenizer y lo añade a tokenized corpus

bm25 = BM25Okapi(tokenized_corpus)
# Aquí construyes el modelo/índice BM25 a partir de todo el corpus

100%|██████████| 15/15 [00:00<?, ?it/s]


In [10]:
# Esta función hace ya una búsqueda léxica completa con BM25 y devuelve los mejores textos según coincidencia de términos.

def keyword_search(query, top_k=3, num_candidates=15):
    # query -> La consulta
    # top_k=3-> cuántos resultados quieres mostrar al final.
    # num_candidates=15: cuántos candidatos recuperas inicialmente antes de ordenarlos.
    
    print("Input question:", query)

    ##### BM25 search (lexical search) #####
    bm25_scores = bm25.get_scores(bm25_tokenizer(query))
    # tokeniza la query con la misma función usada para los documentos y BM25 calcula un score para cada texto del corpus.
    top_n = np.argpartition(bm25_scores, -num_candidates)[-num_candidates:]
    # np.argpartition() no devuelve los scores, sino los índices de los elementos.
    # Lo que hace aquí es seleccionar los índices correspondientes a los num_candidates scores más altos.
    # Como los más cercanos tienen puentuaciones más altas ponemos en negativo num_candidates -> Para empezar por atrás -> Números + altos
    
    bm25_hits = [{'corpus_id': idx, 'score': bm25_scores[idx]} for idx in top_n]
    # Crea una estructura con los  candidatos.
    # top_n -> ínideces de mejores candidatos pero todavía desordenados
    
    bm25_hits = sorted(bm25_hits, key=lambda x: x['score'], reverse=True)
    # ordenamos por score
    # «Para cada elemento x de bm25_hits, usa x['score'] como criterio de ordenación»
    # reverse=True -> Ordenado de mayor a menor -> De mejor a peor score BM25

    print(f"Top-{top_k} lexical search (BM25) hits")
    for hit in bm25_hits[0:top_k]: # recorre solo los primeros top_k resultados.
        print("\t{:.3f}\t{}".format(hit['score'], texts[hit['corpus_id']].replace("\n", " ")))
        # {:.3f} -> muestra este número con 3 decimales
        # \t - tabuladores para separar visualmente las columnas


In [11]:
keyword_search(query = "how precise was the science")

Input question: how precise was the science
Top-3 lexical search (BM25) hits
	1.789	Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan
	1.373	Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar
	0.000	It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine


## Caveats of Dense Retrieval


In [12]:
query = "What is the mass of the moon?"
results = search(query)
results

Query:'What is the mass of the moon?'
Nearest neighbors:


,id,texts,distance
0,12,It has also received praise from many astronom...,1.468251
1,4,Caltech theoretical physicist and 2017 Nobel l...,1.549838
2,13,"Since its premiere, Interstellar gained a cult...",1.550955


# Reranking Example


In [13]:
# Hacemos rerankinf con Cohere
query = "how precise was the science"

results = co.rerank(query=query, documents=texts, top_n=3, return_documents=True)
# Tengo esta pregunta y estos documentos. Evalúa qué documentos responden mejor a la pregunta y devuélveme los 3 mejores.
# return_documents=True -> además de puntuaciones e índices, quieres que Cohere te devuelva también el texto del documento.
# en la versión de la API actual hay que definir tb el modelo -> model="rerank-v4.0-pro"

results.results
# accedes a la lista de resultados que contiene la respuesta de Cohere.

[RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics'), index=12, relevance_score=0.15232232),
 RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014'), index=10, relevance_score=0.050354082),
 RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan'), index=0, relevance_score=0.0350424)]

In [14]:
# Recorre los resultados del reranker y los imprime

for idx, result in enumerate(results.results):
    print(idx, result.relevance_score , result.document.text)

# results.results contiene los 3 resultados que pedimos con top_n=3
# enumerate() hace que en cada iteración tengamos dos cosas: posición (idx) y result (resultado de Cohere)

0 0.15232232 It has also received praise from many astronomers for its scientific accuracy and portrayal of theoretical astrophysics
1 0.050354082 The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014
2 0.0350424 Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan


In [15]:
def keyword_and_reranking_search(query, top_k=3, num_candidates=10):
    print("Input question:", query)

    ##### BM25 search (lexical search) #####
    bm25_scores = bm25.get_scores(bm25_tokenizer(query))
    top_n = np.argpartition(bm25_scores, -num_candidates)[-num_candidates:]
    bm25_hits = [{'corpus_id': idx, 'score': bm25_scores[idx]} for idx in top_n]
    bm25_hits = sorted(bm25_hits, key=lambda x: x['score'], reverse=True)

    print(f"Top-3 lexical search (BM25) hits")
    for hit in bm25_hits[0:top_k]:
        print("\t{:.3f}\t{}".format(hit['score'], texts[hit['corpus_id']].replace("\n", " ")))

    #Add re-ranking
    docs = [texts[hit['corpus_id']] for hit in bm25_hits]

    print(f"\nTop-3 hits by rank-API ({len(bm25_hits)} BM25 hits re-ranked)")
    results = co.rerank(query=query, documents=docs, top_n=top_k, return_documents=True)
    for hit in results.results:
        print("\t{:.3f}\t{}".format(hit.relevance_score, hit.document.text.replace("\n", " ")))

In [16]:
keyword_and_reranking_search(query = "how precise was the science")

Input question: how precise was the science
Top-3 lexical search (BM25) hits
	1.789	Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan
	1.373	Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of Interstellar
	0.000	Interstellar uses extensive practical and miniature effects and the company Double Negative created additional digital effects

Top-3 hits by rank-API (10 BM25 hits re-ranked)
	0.035	Interstellar is a 2014 epic science fiction film co-written, directed, and produced by Christopher Nolan
	0.032	It stars Matthew McConaughey, Anne Hathaway, Jessica Chastain, Bill Irwin, Ellen Burstyn, Matt Damon, and Michael Caine
	0.031	Caltech theoretical physicist and 2017 Nobel laureate in Physics[4] Kip Thorne was an executive producer, acted as a scientific consultant, and wrote a tie-in book, The Science of In

# Retrieval-Augmented Generation

## Example: Grounded Generation with an LLM API


In [17]:
# Construimos nuestro primer RAG completo
# El código es corto paruqe ya construí las piezas anteriores

query = "income generated"

# 1- Retrieval
# We'll use embedding search. But ideally we'd do hybrid
results = search(query)

# "income generated" -> Cohere embedding -> vector de la query -> FAISS -> 3 chunks + cercanos
# results era el DF ( text       distance) 

# 2- Grounded Generation
docs_dict = [{'text': text} for text in results['texts']]
# crea una lista con los distintos textos con la estructura que necesita Cohere

response = co.chat(
    message = query,
    documents=docs_dict
)

# Aquí Cohere recibe dos cosas distintas:
    # message -> "income generated" (pregunta)
    # documents -> Los chunks recuperados por FAISS
# El LLM puede entonces generar la respuesta utilizando esos documentos.

print(response.text)
# Muestra la resouesta generada

Query:'income generated'
Nearest neighbors:
The film generated $677 million worldwide and $773 million with subsequent re-releases.


In [18]:
response

NonStreamedChatResponse(text='The film generated $677 million worldwide and $773 million with subsequent re-releases.', generation_id='6d18385f-568e-4c9c-b137-19e4f77334d6', citations=[ChatCitation(start=19, end=41, text='$677 million worldwide', document_ids=['doc_0'], type='TEXT_CONTENT'), ChatCitation(start=46, end=87, text='$773 million with subsequent re-releases.', document_ids=['doc_0'], type='TEXT_CONTENT')], documents=[{'id': 'doc_0', 'text': 'The film had a worldwide gross over $677 million (and $773 million with subsequent re-releases), making it the tenth-highest grossing film of 2014'}], is_search_required=None, search_queries=None, search_results=None, finish_reason='COMPLETE', tool_calls=None, chat_history=[Message_User(message='income generated', tool_calls=None, role='USER'), Message_Chatbot(message='The film generated $677 million worldwide and $773 million with subsequent re-releases.', tool_calls=None, role='CHATBOT')], prompt=None, meta=ApiMeta(api_version=ApiMetaA

In [ ]:
response.citations

[ChatCitation(start=21, end=57, text='worldwide gross of over $677 million', document_ids=['doc_0']),
 ChatCitation(start=62, end=103, text='$773 million with subsequent re-releases.', document_ids=['doc_0'])]

## Example: RAG with Local Models


### Loading the Generation Model


In [ ]:
#!wget https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-q4.gguf

--2024-06-21 09:49:22--  https://huggingface.co/lmstudio-community/Phi-3-mini-4k-instruct-GGUF/resolve/main/Phi-3-mini-4k-instruct-Q8_0.gguf
Resolving huggingface.co (huggingface.co)... 18.164.174.118, 18.164.174.23, 18.164.174.17, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.118|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://cdn-lfs-us-1.huggingface.co/repos/8e/3f/8e3fafa0351929e621a3db9a53b131a9d7f4b222332208032555bb92f11ab100/8d2f3732e31c354e169cd81dcde9807a1c73b85b9a0f9b16c19013e7a4bb151c?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27Phi-3-mini-4k-instruct-Q8_0.gguf%3B+filename%3D%22Phi-3-mini-4k-instruct-Q8_0.gguf%22%3B&Expires=1719222562&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTcxOTIyMjU2Mn19LCJSZXNvdXJjZSI6Imh0dHBzOi8vY2RuLWxmcy11cy0xLmh1Z2dpbmdmYWNlLmNvL3JlcG9zLzhlLzNmLzhlM2ZhZmEwMzUxOTI5ZTYyMWEzZGI5YTUzYjEzMWE5ZDdmNGIyMjIzMzIyMDgwMzI1NTViYjkyZjExYWIxMDAvOGQyZ

In [19]:
from pathlib import Path
import wget

model_path = Path(r"D:\AI\HuggingFace\Phi-3-mini-4k-instruct-fp16.gguf")

url = "https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf"

if model_path.exists():
    print("El modelo ya está descargado.")
else:
    print("Descargando modelo...")
    wget.download(url, str(model_path))
    print("\nDescarga completada.")


# Hago esta lógica para que no descargue el modelo con wget de nuevo ya que cambié el fichero a otra carpeta    
# If this command does not work for you, you can use the link directly to download the model
# https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf

El modelo ya está descargado.


In [22]:

from langchain import LlamaCpp

# Make sure the model path is correct for your system!
llm = LlamaCpp(
    model_path=r"D:\AI\HuggingFace\Phi-3-mini-4k-instruct-fp16.gguf", # dónde está el modelo físicamente
    n_gpu_layers=-1,
    max_tokens=500,
    n_ctx=2048,
    seed=42,
    verbose=False
)

Va a reconstruir el RAG con modelos locales en vez de depender de Cohere.

Va a repetir el pipeline que ya hice antes:

```text
DOCUMENTOS
    ↓
BGE-small
    ↓
EMBEDDINGS
    ↓
índice / vector store
    ↓
   QUERY
    ↓
embedding query
    ↓
RETRIEVAL
    ↓
documentos relevantes
    ↓
query + documentos
    ↓
Phi-3
    ↓
RESPUESTA
```

### Loading the Embedding Model

In [25]:
from langchain.embeddings.huggingface import HuggingFaceEmbeddings

# Embedding Model for converting text to numerical representations
embedding_model = HuggingFaceEmbeddings(
    model_name='BAAI/bge-small-en-v1.5'
)
# Cargamos el modelo de embeddings.
# Este model produce embeddings de 384 dimensiones. Además, es bastante pequeño: unos 33,4 millones de parámetros,
# y admite secuencias de hasta 512 tokens

C:\Users\srmjf\AppData\Local\Temp\ipykernel_20108\1723602048.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 0.3.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embedding_model = HuggingFaceEmbeddings(
C:\Users\srmjf\anaconda3\envs\thellmbook\lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\srmjf\anaconda3\envs\thellmbook\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in D:\AI\HuggingFace\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Preparing the Vector Database

In [26]:
from langchain.vectorstores import FAISS
# importa la integración de FAISS dentro de LangChain.

# Create a local vector database
db = FAISS.from_texts(texts, embedding_model)

C:\Users\srmjf\anaconda3\envs\thellmbook\lib\site-packages\transformers\models\bert\modeling_bert.py:435: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:455.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


Antes utilicé **FAISS directamente**:

```python
import faiss

index = faiss.IndexFlatL2(dim)
index.add(np.float32(embeds))
```

```text
embeddings
    ↓
FAISS
    ↓
IndexFlatL2
    ↓
index.add(...)
    ↓
índice vectorial
```

Aquí tú mismo creabas y gestionabas el índice FAISS.

### The RAG Prompt


In [27]:
from langchain import PromptTemplate
# PromptTemplate sirve para construir el prompt que recibirá Phi-3.

from langchain.chains import RetrievalQA
# RetrievalQA es la cadena de LangChain que conecta retrieval + generación.


# Create a prompt template
template = """<|user|>
Relevant information:
{context}

Provide a concise answer the following question using the relevant information provided above:
{question}<|end|>
<|assistant|>"""
prompt = PromptTemplate(
    template=template,
    input_variables=["context", "question"]
)

# RAG Pipeline
rag = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type='stuff',
    retriever=db.as_retriever(),
    chain_type_kwargs={
        "prompt": prompt
    },
    verbose=True
)
# construye el pipeline
# db.as_retriever() -> le dice a LangChain que utilice ese índice como retriever
# chain_type='stuff' -> LangChain toma los documentos recuperados y los "mete" todos juntos en el contexto del prompt.

{context}   ← documentos recuperados por FAISS

{question}  ← pregunta del usuario

```text
                   QUESTION
                      │
                      ↓
              BGE-small-en-v1.5
                 embedding
                      │
                      ↓
                    FAISS
                      │
                nearest neighbors
                      │
                      ↓
             documentos relevantes
                      │
                      ↓
              PromptTemplate
             ┌────────┴─────────┐
             │                  │
        {context}          {question}
             │                  │
             └────────┬─────────┘
                      ↓
                    Phi-3
                      ↓
                  RESPUESTA
```

In [30]:
rag.invoke('Revenue generated')
# Llamamos al modelo y hacemos las pregunta



> Entering new RetrievalQA chain...

> Finished chain.


{'query': 'Revenue generated',
 'result': ' The Interstellar film generated over $677 million worldwide in its original release, and this figure increased to approximately $773 million with subsequent re-releases.'}